# Part 2: Analysis and Comparison of Pipe Performance

---

## 1. Theoretical Analysis

### Mechanism and Data Transfer
**Unnamed Pipes:**
* **Mechanism:** Created via the `pipe()` system call. They rely entirely on kernel memory buffers.
* **Connection:** They function through file descriptor inheritance. A parent process creates the pipe and then `fork()`s; the child inherits the open file descriptors, allowing communication. They are restricted to related processes (parent-child).

**Named Pipes (FIFOs):**
* **Mechanism:** Created via `mkfifo` and exist as a special device file within the file system.
* **Connection:** Processes connect to them by opening the file path using standard `open()` calls. This allows unrelated processes (no parent-child relationship) to communicate as long as they have access to the filesystem path.

### Performance Differences
The primary factors causing performance differences are **Filesystem Operations** and **System Calls**:
* **Named Pipes:** Require filesystem metadata operations (creating inodes, directory lookups, checking permissions) every time they are created or opened. This increases System Time and CPU overhead.
* **Unnamed Pipes:** Exist only as anonymous resources in the kernel. They bypass the filesystem layer entirely, resulting in fewer context switches and lower overhead.

### Use Cases and Trade-offs
* **When to use Named Pipes:** Best when processes are unrelated (not in the same process tree) or when the communication channel needs to persist beyond the life of the creating process. They offer greater flexibility but require synchronization (reader/writer matching) and filesystem cleanup.
* **When to use Unnamed Pipes:** Best for linear parent-child data flow (like shell pipelines `|`). They provide automatic cleanup (destroyed when processes exit) and higher performance but lack flexibility for non-related processes.

---

## 2. Practical Comparison (Benchmark Analysis)

### Benchmark Results
The following data was collected using `benchmark.sh` running `named.sh` and `unnamed.sh` over 10 iterations.

| Metric | Named Pipe (`named.sh`) | Unnamed Pipe (`unnamed.sh`) |
| :--- | :---: | :---: |
| **Average User Time** | 1.081 s | 0.657 s |
| **Average System Time** | 3.544 s | 2.256 s |
| **Average CPU %** | 93.50 % | **145.20 %** |
| **Average Elapsed Time** | 4.923 s | **2.001 s** |

### Analysis of Results

**1. Elapsed Time Discrepancy**
The Unnamed Pipe implementation was approximately **2.5x faster** (2.001s vs 4.923s) than the Named Pipe implementation.
* **Reason:** The `named.sh` script forces the OS to perform expensive filesystem operations (`mkfifo` to create the file and `rm` to delete it) inside a loop 1000 times. The `unnamed.sh` script performs all operations in memory, avoiding disk metadata overhead.

**2. CPU Usage and Parallelism (The 145% Figure)**
A significant difference is observed in CPU utilization, with Unnamed Pipes reaching **145.20%**.
* **Reason:** A value over 100% indicates that the process utilized multiple CPU cores simultaneously. This occurs because the shell `fork()`s both the writer (`dd`) and the reader (`cat`) processes almost simultaneously. They operate under the **Producer-Consumer model**: as soon as `dd` writes data to the kernel buffer, `cat` reads it immediately without waiting for the writer to finish.
* **Contrast:** In `named.sh` (93.50%), the process failed to fully utilize even a single core. The script introduces serial bottlenecks (creating and deleting files sequentially) which interrupts the data flow, preventing the writer and reader from achieving effective parallelism.

**3. User Time Analysis**
The User Time for `named.sh` (1.081s) was higher than `unnamed.sh` (0.657s).
* **Reason:** User Time measures the time the CPU spends executing code in user space (shell interpretation and library calls). The `named.sh` script is more complex to interpret: it explicitly executes `mkfifo` and `rm` commands and manages background subshells `(...) &` in every iteration. The `unnamed.sh` script uses the native pipe operator `|`, which is syntactically simpler and optimized within the shell, resulting in less user-mode overhead for the interpreter.

**4. System Time Overhead**
The Named Pipe required significantly higher System Time (3.544s vs 2.256s).
* **Reason:** This confirms the theoretical expectation that managing filesystem nodes (inodes) via `mkfifo` and `rm` requires more kernel intervention and CPU cycles than managing simple memory buffers used by unnamed pipes.

### Conclusion
The benchmark confirms that **Unnamed Pipes** are significantly more efficient for related processes due to the lack of filesystem I/O overhead and better support for parallelism. **Named Pipes** incur a performance penalty due to filesystem management but remain necessary for communication between unrelated processes.